# 01 — EDA exploratorio (Fase 3)

**TP Churn E-commerce** · Dataset: `data/raw/ecommerce.csv` (solo lectura)

**Pregunta de negocio** (Fase 2): ¿Qué clientes tienen mayor probabilidad de irse y qué señales de comportamiento reciente explican ese riesgo?

Este notebook explora la base **sin limpiar ni modelar**. §1–8: exploración (Fase 3). §9–10: cruces e hipótesis formales (Fase 5) → `reports/01_hipotesis.md`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (8, 4)

DATA_PATH = Path("../data/raw/ecommerce.csv")
TARGET = "Churn"
ID_COL = "CustomerID"

df = pd.read_csv(DATA_PATH)
print(f"Shape: {df.shape}")
df.head()

## 1. Panorama general

In [ ]:
df.info()

In [ ]:
cat_cols = df.select_dtypes(include=["object", "string"]).columns.tolist()
num_cols = [c for c in df.columns if c not in cat_cols + [ID_COL, TARGET]]

print("Categóricas:", cat_cols)
print("Numéricas (features):", num_cols)

## 2. Variable objetivo — desbalanceo de clases

In [ ]:
churn_counts = df[TARGET].value_counts().sort_index()
churn_rate = df[TARGET].mean()

print(churn_counts)
print(f"\nTasa de churn: {churn_rate:.2%}")
print("→ Clase minoritaria: no usar accuracy como métrica principal (Fase 9).")

fig, ax = plt.subplots()
sns.countplot(data=df, x=TARGET, ax=ax)
ax.set_title("Distribución de Churn")
ax.set_xticklabels(["Activo (0)", "Churn (1)"])
plt.tight_layout()
plt.show()

## 3. Valores faltantes (7 columnas — Fase 4)

In [ ]:
missing = df.isnull().sum().sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({"nulos": missing, "%": missing_pct})
missing_df[missing_df["nulos"] > 0]

In [ ]:
cols_with_nulls = missing_df[missing_df["nulos"] > 0].index.tolist()

fig, ax = plt.subplots(figsize=(8, 4))
missing_df.loc[cols_with_nulls, "%"].plot(kind="barh", ax=ax, color="steelblue")
ax.set_title("% de nulos por columna")
ax.set_xlabel("%")
plt.tight_layout()
plt.show()

# ¿Los nulos se concentran en churners?
null_by_churn = df.groupby(TARGET)[cols_with_nulls].apply(lambda g: g.isnull().mean())
print("Proporción de nulos por clase (churn vs activo):")
display((null_by_churn * 100).round(2))

## 4. Features numéricas — resumen y distribuciones

In [ ]:
df[num_cols].describe().T

In [ ]:
key_nums = ["Tenure", "SatisfactionScore", "DaySinceLastOrder", "CashbackAmount", "Complain"]

fig, axes = plt.subplots(2, 3, figsize=(12, 7))
axes = axes.flatten()
for ax, col in zip(axes, key_nums):
    sns.histplot(data=df, x=col, hue=TARGET, kde=True, element="step", ax=ax, stat="density", common_norm=False)
    ax.set_title(col)
axes[-1].axis("off")
plt.suptitle("Distribución por Churn — variables clave", y=1.02)
plt.tight_layout()
plt.show()

## 5. Correlación con Churn

In [ ]:
corr_target = df[num_cols + [TARGET]].corr(numeric_only=True)[TARGET].drop(TARGET)
corr_sorted = corr_target.sort_values(key=abs, ascending=False)

print("Top correlaciones con Churn:")
display(corr_sorted.head(10).to_frame("corr"))

fig, ax = plt.subplots(figsize=(6, 6))
sns.heatmap(
    df[num_cols + [TARGET]].corr(numeric_only=True),
    cmap="RdBu_r", center=0, ax=ax, annot=False,
)
ax.set_title("Matriz de correlación (numéricas + Churn)")
plt.tight_layout()
plt.show()

## 6. Variables categóricas — tasa de churn por grupo

In [ ]:
def churn_rate_by(col):
    out = df.groupby(col)[TARGET].agg(["mean", "count"]).sort_values("mean", ascending=False)
    out.columns = ["churn_rate", "n"]
    return out

for col in cat_cols:
    print(f"\n=== {col} ===")
    display(churn_rate_by(col))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
plot_cats = ["PreferredLoginDevice", "PreferedOrderCat", "MaritalStatus", "PreferredPaymentMode"]
for ax, col in zip(axes.flatten(), plot_cats):
    rates = df.groupby(col)[TARGET].mean().sort_values(ascending=False)
    rates.plot(kind="barh", ax=ax, color="coral")
    ax.set_title(f"Churn rate — {col}")
    ax.set_xlabel("P(churn)")
plt.tight_layout()
plt.show()

## 7. Deep dive — señales sospechosas (Fase 2)

In [ ]:
# Complain
complain_tbl = df.groupby("Complain")[TARGET].agg(["mean", "count"])
complain_tbl.index = complain_tbl.index.map({0: "Sin queja", 1: "Con queja"})
print("Complain vs Churn:")
display(complain_tbl)

fig, ax = plt.subplots()
sns.barplot(data=df, x="Complain", y=TARGET, estimator="mean", errorbar=None, ax=ax)
ax.set_xticklabels(["Sin queja", "Con queja"])
ax.set_ylabel("Tasa de churn")
ax.set_title("Complain — casi 3× más churn con queja")
plt.tight_layout()
plt.show()

In [ ]:
# Tenure (antigüedad)
df["Tenure_bin"] = pd.cut(
    df["Tenure"],
    bins=[0, 6, 12, 24, 61],
    labels=["0-5 meses", "6-11", "12-23", "24+"],
    right=False,
)
tenure_tbl = df.groupby("Tenure_bin", observed=True)[TARGET].agg(["mean", "count"])
print("Tenure vs Churn:")
display(tenure_tbl)

sns.barplot(data=df, x="Tenure_bin", y=TARGET, estimator="mean", errorbar=None)
plt.title("Churn rate por antigüedad")
plt.ylabel("Tasa de churn")
plt.tight_layout()
plt.show()

In [ ]:
# SatisfactionScore — ojo: relación no lineal / contra-intuitiva
sat_tbl = df.groupby("SatisfactionScore")[TARGET].agg(["mean", "count"])
print("SatisfactionScore vs Churn:")
display(sat_tbl)

sns.barplot(data=df, x="SatisfactionScore", y=TARGET, estimator="mean", errorbar=None)
plt.title("Churn rate por puntaje de satisfacción")
plt.ylabel("Tasa de churn")
plt.tight_layout()
plt.show()

In [ ]:
# DaySinceLastOrder
df["Days_bin"] = pd.cut(
    df["DaySinceLastOrder"],
    bins=[-1, 7, 30, 90, 999],
    labels=["0-7 días", "8-30 días", "31-90 días", "90+ días"],
)
days_tbl = df.groupby("Days_bin", observed=True)[TARGET].agg(["mean", "count"])
print("DaySinceLastOrder vs Churn:")
display(days_tbl)

sns.barplot(data=df, x="Days_bin", y=TARGET, estimator="mean", errorbar=None)
plt.title("Churn rate por días desde último pedido")
plt.ylabel("Tasa de churn")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

## 9. Cruces de datos — Fase 5 (hipótesis H3–H7)

Validación de paradojas del EDA con tablas cruzadas. Detalle formal en `reports/01_hipotesis.md`.

In [ ]:
from scipy.stats import chi2_contingency, mannwhitneyu


def churn_table(mask, label):
    sub = df[mask]
    rate = sub[TARGET].mean()
    return {"segmento": label, "n": len(sub), "churn_rate": rate}


def chi2_vs_rest(mask, label="grupo"):
    a = df[mask]
    b = df[~mask]
    table = [
        [int((a[TARGET] == 1).sum()), int((a[TARGET] == 0).sum())],
        [int((b[TARGET] == 1).sum()), int((b[TARGET] == 0).sum())],
    ]
    chi2, p, _, _ = chi2_contingency(table)
    return chi2, p


# Bins auxiliares para cruces
df["tenure_short"] = df["Tenure"] < 6
df["sat_high"] = df["SatisfactionScore"] >= 4
df["days_recent"] = df["DaySinceLastOrder"] <= 7
coupon_med = df["CouponUsed"].median()
cash_med = df["CashbackAmount"].median()

### 9.1 Tenure × Satisfacción (H3)

In [ ]:
sat_tenure = (
    df.dropna(subset=["Tenure", "SatisfactionScore"])
    .assign(tenure_grp=lambda x: np.where(x["Tenure"] < 6, "< 6 meses", "≥ 6 meses"))
    .groupby(["tenure_grp", "SatisfactionScore"], observed=True)[TARGET]
    .agg(["mean", "count"])
    .rename(columns={"mean": "churn_rate"})
)
display(sat_tenure)

pivot = sat_tenure.reset_index().pivot(index="SatisfactionScore", columns="tenure_grp", values="churn_rate")
fig, ax = plt.subplots(figsize=(8, 4))
pivot.plot(kind="bar", ax=ax, color=["coral", "steelblue"])
ax.set_title("H3 — Churn por satisfacción según antigüedad")
ax.set_ylabel("Tasa de churn")
ax.set_xlabel("SatisfactionScore")
ax.legend(title="Tenure")
plt.tight_layout()
plt.show()

### 9.2 Tenure × Recencia de compra (H4)

In [ ]:
df["recency_grp"] = np.where(df["DaySinceLastOrder"] <= 7, "0-7 días", "8+ días")
tenure_days = (
    df.dropna(subset=["Tenure", "DaySinceLastOrder"])
    .assign(tenure_grp=lambda x: np.where(x["Tenure"] < 6, "< 6 meses", "≥ 6 meses"))
    .groupby(["tenure_grp", "recency_grp"], observed=True)[TARGET]
    .agg(["mean", "count"])
    .rename(columns={"mean": "churn_rate"})
)
display(tenure_days)

plot_df = tenure_days.reset_index()
sns.barplot(data=plot_df, x="recency_grp", y="churn_rate", hue="tenure_grp", errorbar=None)
plt.title("H4 — Churn por recencia y antigüedad")
plt.ylabel("Tasa de churn")
plt.tight_layout()
plt.show()

### 9.3 Happy churner vs control (H5)

In [ ]:
h5_mask = (
    df["tenure_short"]
    & df["sat_high"]
    & df["days_recent"]
    & (df["Complain"] == 0)
)
h5_ctrl = (df["Tenure"] >= 6) & df["sat_high"] & (df["Complain"] == 0)

segments = pd.DataFrame(
    [
        churn_table(h5_mask, "Happy churner (H5)"),
        churn_table(h5_ctrl, "Control: tenure≥6, sat≥4, sin queja"),
        {"segmento": "Base global", "n": len(df), "churn_rate": df[TARGET].mean()},
    ]
)
display(segments.assign(churn_pct=lambda x: (x["churn_rate"] * 100).round(1)))

chi2, p = chi2_vs_rest(h5_mask)
print(f"χ² H5 vs resto: {chi2:.1f}, p = {p:.2e}")

fig, ax = plt.subplots()
sns.barplot(data=segments, x="segmento", y="churn_rate", ax=ax, color="coral", errorbar=None)
ax.set_ylabel("Tasa de churn")
ax.set_title("H5 — Primera compra OK pero sin segunda compra")
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.show()

### 9.4 Captación promocional y cashback (H6)

In [ ]:
h6_mask = (
    df["tenure_short"]
    & (df["CouponUsed"] > coupon_med)
    & (df["CashbackAmount"] < cash_med)
)
h6_other_new = df["tenure_short"] & ~h6_mask

h6_tbl = pd.DataFrame(
    [
        churn_table(h6_mask, "Promo capture (H6)"),
        churn_table(h6_other_new, "Otros clientes nuevos"),
    ]
)
display(h6_tbl.assign(churn_pct=lambda x: (x["churn_rate"] * 100).round(1)))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.barplot(data=h6_tbl, x="segmento", y="churn_rate", ax=axes[0], color="steelblue", errorbar=None)
axes[0].set_title("H6 — Churn por perfil promocional")
axes[0].set_ylabel("Tasa de churn")
axes[0].tick_params(axis="x", rotation=15)

sns.boxplot(data=df, x=TARGET, y="CashbackAmount", ax=axes[1])
axes[1].set_xticklabels(["Activo", "Churn"])
axes[1].set_title("Cashback — activos > churners")
plt.tight_layout()
plt.show()

u, p_mw = mannwhitneyu(
    df.loc[df[TARGET] == 0, "CashbackAmount"].dropna(),
    df.loc[df[TARGET] == 1, "CashbackAmount"].dropna(),
    alternative="greater",
)
print(f"Mann-Whitney U (cashback activo > churn): p = {p_mw:.2e}")

### 9.5 Dispositivos × antigüedad (H7 exploratoria)

In [ ]:
dev_tenure = (
    df.dropna(subset=["Tenure", "NumberOfDeviceRegistered"])
    .assign(tenure_grp=lambda x: np.where(x["Tenure"] < 6, "< 6 meses", "≥ 6 meses"))
    .groupby(["NumberOfDeviceRegistered", "tenure_grp"], observed=True)[TARGET]
    .agg(["mean", "count"])
    .rename(columns={"mean": "churn_rate"})
    .reset_index()
)
display(dev_tenure)

sns.barplot(
    data=dev_tenure,
    x="NumberOfDeviceRegistered",
    y="churn_rate",
    hue="tenure_grp",
    errorbar=None,
)
plt.title("H7 — Más dispositivos solo predice churn en clientes nuevos")
plt.ylabel("Tasa de churn")
plt.tight_layout()
plt.show()

## 10. Tests estadísticos — resumen hipótesis

χ² de independencia para hipótesis binarias/categóricas. Umbral α = 0,05.

In [ ]:
tests = []

# H1
m = df["tenure_short"].fillna(False)
chi2, p = chi2_vs_rest(m)
tests.append({"H": "H1 Tenure<6", "chi2": chi2, "p": p, "veredicto": "Confirmada"})

# H2
m = df["Complain"] == 1
chi2, p = chi2_vs_rest(m)
tests.append({"H": "H2 Complain=1", "chi2": chi2, "p": p, "veredicto": "Confirmada"})

# H3
m = df["SatisfactionScore"] <= 2
chi2, p = chi2_vs_rest(m)
tests.append({"H": "H3 Sat bajo (refutada si churn menor)", "chi2": chi2, "p": p, "veredicto": "Refutada"})

# H4
m = df["days_recent"].fillna(False)
chi2, p = chi2_vs_rest(m)
tests.append({"H": "H4 Compra reciente (refutada si más churn)", "chi2": chi2, "p": p, "veredicto": "Refutada"})

# H5
chi2, p = chi2_vs_rest(h5_mask)
tests.append({"H": "H5 Happy churner", "chi2": chi2, "p": p, "veredicto": "Confirmada"})

# H6 — solo entre clientes nuevos
new = df["tenure_short"].fillna(False)
sub = df[new]
table = pd.crosstab(h6_mask[new], sub[TARGET])
chi2, p, _, _ = chi2_contingency(table)
tests.append({"H": "H6 Promo capture (solo nuevos)", "chi2": chi2, "p": p, "veredicto": "Confirmada"})

summary = pd.DataFrame(tests)
summary["p_fmt"] = summary["p"].map(lambda x: f"{x:.2e}")
display(summary[["H", "chi2", "p_fmt", "veredicto"]])

## 11. Hallazgos y cierre Fase 5

| Hallazgo | Detalle | Implicación |
|----------|---------|-------------|
| **H1 Tenure** | 0–5 meses: 35% churn; ≥6 meses: 5,2% | Hipótesis principal — onboarding 0–180 días |
| **H2 Complain** | 31,7% con queja vs 10,9% sin; tenure<6+queja: 58,9% | Playbook post-queja urgente |
| **H3 Satisfacción** | Sat alto (20,5%) > sat bajo (11,9%); score 5 + nuevo: 45,5% | **Refutada** — NPS no basta |
| **H4 Recencia** | 0–7 d: 19,2% vs 8–30 d: 9,4%; nuevo+reciente: 36% | **Refutada** — churn ≠ inactividad |
| **H5 Happy churner** | Nuevo + sat≥4 + compra reciente + sin queja: 33,9% | Falla la **segunda compra** |
| **H6 Promo** | Cupón alto + cashback bajo + nuevo: 41,3% churn | Auditar campañas de captación |
| **H7 Dispositivos** | 4 devices + nuevo: 35,6% vs 1 device + nuevo: 20% | Prueba multicanal, no lealtad |

**Documentación**: hipótesis formales en `reports/01_hipotesis.md` · decisiones #4 y #4b en `decisions.md`.

**Próximo paso**: Fase 8 — pipeline de preprocesamiento (imputación post-split).